# Stage 2 — Instruction Fine-Tuning (Supervised Fine-Tuning / SFT)
### Pull Stage 1 from the Hub → teach it to answer → push Stage 2

**Notebook 2 of 4.**

### What this stage does, in plain words

Stage 1 gave the model the **vocabulary** of HR. But it still doesn't know how to
*behave*: ask it a question and it will just ramble on continuing your sentence.

Here we show it hundreds of **(question, ideal answer)** pairs so it learns the
pattern: *when someone asks, produce a clean answer and then stop.*

### Why is this called "supervised"?

The training engine is **identical** to Stage 1 — still next-token prediction,
still cross-entropy loss. What changed is the **data**:

| | Stage 1 | Stage 2 (this) |
|---|---|---|
| Data | raw text | (question → ideal answer) pairs |
| Who made the labels? | nobody — the next word *is* the label | **a human wrote the ideal answer** |
| Name | self-supervised | **supervised** |

That's the whole distinction. A human-provided target = supervised learning. SFT
is just supervised learning delivered through the next-token interface.

## 1. Install libraries

In [1]:
# ============================================================
# Step 1. Install libraries
# ============================================================
!pip install -q unsloth
!pip install -q transformers trl datasets peft bitsandbytes accelerate sentencepiece protobuf huggingface_hub

## 2. Imports and GPU check

In [2]:
# ============================================================
# Step 2. Imports + GPU check
# ============================================================
import torch, json, gc
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda
GPU: Tesla T4


## Hugging Face login

Every stage pushes its merged model to the Hub, and the next stage pulls it back
down. So we log in once, here, at the top.

**Never paste a raw token into a notebook cell.** A token in a saved `.ipynb` is
a leaked credential — anyone with the file can push or delete on your account.
Use a **Colab secret** instead:

> Click the **key icon (🔑)** in the Colab left sidebar → **Add new secret** →
> Name: `HF_TOKEN`, Value: your token from
> https://huggingface.co/settings/tokens (needs **write** access) → toggle
> **Notebook access** on.

The cell below reads that secret automatically, and falls back to an interactive
prompt if it isn't set.

In [3]:
# ============================================================
# Hugging Face login
# ============================================================
from huggingface_hub import login, whoami

try:
    # Colab secret named HF_TOKEN (recommended).
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    # Fallback: paste the token when prompted.
    login()

HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Logged in as: Mohan143


## 3. Configuration

Note `learning_rate = 1e-4` — **half** of Stage 1. We're refining a model that
already knows the domain, so we take gentler steps to avoid washing out what
Stage 1 taught it.

In [4]:
# ============================================================
# Step 3. Configuration
# ============================================================
# --- Base model = Stage 1's output, pulled from the Hub ---
STAGE1_REPO = f"{HF_USERNAME}/hr-policy-assistant-stage1"   # <- input
STAGE2_REPO = f"{HF_USERNAME}/hr-policy-assistant-stage2"   # <- output
PRIVATE     = True

model_name     = STAGE1_REPO
max_seq_length = 512
dtype          = None
load_in_4bit   = True      # quantize the Stage-1 model for training

# --- LoRA (a FRESH adapter for instruction-following) ---
lora_rank      = 16
lora_alpha     = 32
lora_dropout   = 0
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# --- Training ---
learning_rate               = 1e-4   # LOWER than Stage 1: refining, not reshaping
num_train_epochs            = 3
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
warmup_steps                = 30
logging_steps               = 20
save_steps                  = 100
seed                        = 42

print(f"Loading base from : {STAGE1_REPO}")
print(f"Will push to      : {STAGE2_REPO}")

Loading base from : Mohan143/hr-policy-assistant-stage1
Will push to      : Mohan143/hr-policy-assistant-stage2


### Concept: quantization (what `load_in_4bit` actually does)

A model's weights are numbers. **Precision** is how many bits we use per number.

| Precision | Bits/weight | Memory for a 1.5B model | Note |
|---|---|---|---|
| float32 (full) | 32 | ~6.0 GB | Original training precision |
| float16 / bfloat16 (half) | 16 | ~3.0 GB | Standard for inference |
| **4-bit (NF4)** | **4** | **~0.9 GB** | What we use — fits a free T4 easily |

**Quantization** = storing those weights in fewer bits. It's like saving a photo
as a smaller JPEG: slightly less detail, dramatically less space.

- **NF4** ("4-bit NormalFloat") is a 4-bit format designed for the bell-curve
  shape that neural-network weights actually follow, so it loses less accuracy
  than naive 4-bit rounding.
- The quantized weights stay **frozen**. Maths still happens in 16-bit, so
  quality loss is small.
- **QLoRA** = *quantized base model* + *LoRA adapter trained on top*. That
  combination is what makes fine-tuning a 1.5B model possible on a free GPU.

Trade-off: 4-bit saves memory but is slightly slower per step and marginally
less precise. For fine-tuning on a T4, it's the right call.

### Concept: LoRA (why we train ~1% of the model)

Full fine-tuning updates **every** weight — expensive, and you must store a whole
new model each time.

**LoRA (Low-Rank Adaptation)** freezes the base model and injects a small pair of
trainable matrices into chosen layers. We train only those.

```text
Output = FrozenBaseWeights(x)  +  (B · A)(x) · (alpha / r)
                                  └─ tiny, trainable ─┘
```

| Argument | Meaning | Why this value |
|---|---|---|
| `r` (rank) | Size/capacity of the adapter | 16 — enough for a domain shift, still tiny |
| `lora_alpha` | Scales the adapter's effect (`alpha/r`) | 32 = 2×rank, a common, stable default |
| `lora_dropout` | Randomly drops adapter units to fight overfitting | 0 — Unsloth's optimized path is fastest at 0 |
| `target_modules` | Which layers get an adapter | All attention (`q,k,v,o_proj`) + MLP (`gate,up,down_proj`) = best quality |
| `use_gradient_checkpointing` | Recompute activations instead of storing them | `"unsloth"` — saves the most VRAM, allows longer sequences |
| `bias` | Whether to train bias terms | `"none"` — standard, keeps adapter small |

Result: trainable parameters drop from ~1.5B to ~18M (about 1%).

## 4. Load and format the instruction data

### Concept: the chat template

The model never sees JSON fields — it only ever sees **text**. So we render each
`(instruction, output)` pair into Qwen2.5's chat format using special marker
tokens:

```text
<|im_start|>system
You are a helpful HR Policy Assistant...<|im_end|>
<|im_start|>user
How many casual leaves can I take per year?<|im_end|>
<|im_start|>assistant
Employees are entitled to 12 days of casual leave per year...<|im_end|>
```

`<|im_start|>` / `<|im_end|>` are **structural tokens** that mark where each turn
begins and ends. They're how the model learns "the user's part stopped, my part
starts here — and here is where I stop."

**The single most important rule in this notebook:** use the **exact same
template at training time and at inference time**. If they differ even slightly,
the model won't recognize the pattern and quality collapses. (Stages 3 and 4 reuse
this identical template for that reason.)

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [5]:
# ============================================================
# Step 4. Load instruction dataset (JSONL: instruction / output)
# ============================================================
DATA_PATH = "/content/drive/MyDrive/AgenticAI/HR-Finetuning/instruction_dataset.jsonl"

instruction_data = []
try:
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            instruction_data.append(json.loads(line))
    print(f"Loaded {len(instruction_data)} instruction examples.")
except FileNotFoundError:
    print("File not found -> using a small embedded sample (replace with your real data).")
    instruction_data = [
        {"instruction": "How many casual leaves can I take per year?",
         "output": "All full-time employees are entitled to 12 days of casual leave per calendar year. Casual leave cannot be carried forward and lapses on December 31st."},
        {"instruction": "What is the work from home policy?",
         "output": "The hybrid work policy allows eligible employees to work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all employees."},
        {"instruction": "How do I apply for sick leave?",
         "output": "Sick leave is granted at 10 days per year for confirmed employees. Notify your immediate supervisor within the first two hours of your shift. Medical documentation is required for absences exceeding three consecutive days."},
        {"instruction": "What is the notice period for resignation?",
         "output": "The notice period is 60 days for roles up to senior manager level and 90 days for director level and above. Buyout requires approval from the department head and HR."},
    ]

print(f"\nSample -> Q: {instruction_data[0]['instruction']}")
print(f"          A: {instruction_data[0]['output'][:150]}...")

Loaded 501 instruction examples.

Sample -> Q: How many casual leaves am I entitled to per year?
          A: All full-time employees are entitled to 12 days of casual leave per calendar year. Casual leave cannot be carried forward to the next year and will la...


In [6]:
# ============================================================
# Step 4b. Apply the Qwen2.5 chat template
# ============================================================
SYSTEM_PROMPT = "You are a helpful HR Policy Assistant. Answer HR-related questions based on company policies."


def format_instruction(example):
    """Render one (instruction, output) pair into Qwen2.5 chat format."""
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return {"text": text}


dataset = Dataset.from_list([format_instruction(ex) for ex in instruction_data])
print(f"Formatted dataset: {len(dataset)} examples\n")
print(dataset[0]["text"][:500])

Formatted dataset: 501 examples

<|im_start|>system
You are a helpful HR Policy Assistant. Answer HR-related questions based on company policies.<|im_end|>
<|im_start|>user
How many casual leaves am I entitled to per year?<|im_end|>
<|im_start|>assistant
All full-time employees are entitled to 12 days of casual leave per calendar year. Casual leave cannot be carried forward to the next year and will lapse on December 31st if unused.<|im_end|>


## 5. Baseline — how does the Stage-1 model answer *before* SFT?

This is the "before" photo. Expect it to ramble, repeat, or continue the question
instead of answering it — because nothing has taught it to answer yet. Keeping
this comparison honest is what makes the "after" meaningful.

In [7]:
# ============================================================
# Step 5. BEFORE training: how does the Stage-1 model respond?
# ============================================================
test_questions = [
    "How many casual leaves can I take per year?",
    "What is the work from home policy?",
    "How do I apply for sick leave?",
    "What benefits does the company provide?",
    "What is the notice period for resignation?",
]


def build_prompt(question):
    """Same template as training, but stop right before the answer."""
    return (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{question}<|im_end|>\n"
            f"<|im_start|>assistant\n")


print("Loading Stage-1 model for the 'before' test...")
test_model, test_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name, max_seq_length=max_seq_length,
    dtype=dtype, load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(test_model)

print("\n" + "="*80)
print("BEFORE STAGE 2 - Stage-1 model (knows HR words, can't answer questions)")
print("="*80)
for q in test_questions:
    inputs = test_tokenizer(build_prompt(q), return_tensors="pt").to("cuda")
    out = test_model.generate(**inputs, max_new_tokens=120, temperature=0.7,
                              do_sample=True, pad_token_id=test_tokenizer.eos_token_id)
    resp = test_tokenizer.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
    print(f"\nQ: {q}\nA: {resp[:220]}")
    print("-"*80)

# Free the VRAM before loading the training copy.
del test_model, test_tokenizer
gc.collect(); torch.cuda.empty_cache()

Loading Stage-1 model for the 'before' test...
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



BEFORE STAGE 2 - Stage-1 model (knows HR words, can't answer questions)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Q: What is the work from home policy?
A: What are the work shift patterns?أشكן work shift patterns are established based on departmental requirements and resource availability. Shift patterns include day shifts, evening shifts, and weekend shifts. Shift flexibi
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How do I apply for sick leave?
A: 
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What benefits does the company provide?
A: What does the employee wellness program offer?iropractic care, nutrition counseling, and mental health support.iropractic care, nutrition counseling, and mental health support are available through the employee wellness 
--------------------------------------------------------------------------------

Q: What is the notice period for resignation?
A: What happens to the earned leave balance when resignation is accepted?处置
isposableleave
How is earned leave calculated?处置
isposableleavebalance
What is the policy regarding outstanding leave balances at the time of resig
--------------------------------------------------------------------------------


## 6. Load the Stage-1 model and attach a fresh LoRA adapter

In [8]:
# ============================================================
# Step 6. Load Stage-1 model (4-bit) + NEW LoRA adapter
# ============================================================
print(f"Loading {model_name} for training...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_name,
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = lora_rank,
    target_modules             = target_modules,
    lora_alpha                 = lora_alpha,
    lora_dropout               = lora_dropout,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = seed,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

Loading Mohan143/hr-policy-assistant-stage1 for training...
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable params: 18,464,768


### Concept: the training arguments

The Stage-2-specific argument is **`packing=False`** — the opposite of Stage 1. Each instruction example is a **self-contained unit**; if we packed them, one Q&A would bleed into the next and the model would learn to run answers together instead of stopping cleanly. Raw text → pack. Instructions → never pack.

| Argument | What it does | Why this value |
|---|---|---|
| `learning_rate` | Step size for each weight update | **1e-4** — half of Stage 1. The model already knows HR; we're teaching behaviour, not rebuilding knowledge. Too high here would erase Stage 1's gains. |
| `num_train_epochs` | How many full passes over the data | 3 — enough to learn, few enough to avoid memorizing |
| `per_device_train_batch_size` | Examples per GPU step | 1 — keeps VRAM low on a T4 |
| `gradient_accumulation_steps` | Accumulate grads over N steps, then update once | 8 → **effective batch = 1×8 = 8** (big-batch stability, small-batch memory) |
| `warmup_steps` | Slowly ramp the LR up at the start | Prevents a large, destabilizing first update |
| `optim="adamw_8bit"` | The optimizer, in 8-bit | Adam stores 2 extra numbers per weight; 8-bit cuts that memory ~4× |
| `weight_decay=0.01` | Mild penalty on large weights | Standard regularization; reduces overfitting |
| `lr_scheduler_type="linear"` | LR decays linearly to 0 | Big steps early, fine-tuning steps late |
| `fp16` / `bf16` | Mixed-precision maths | Auto-picked: `bf16` on newer GPUs (more stable), `fp16` on T4 |
| `seed` | Fixes randomness | 42 — makes the run reproducible |
| `logging_steps` | How often to print the loss | So you can watch it fall |

In [9]:
# ============================================================
# Step 7. SFTTrainer for Stage 2 (instructions, packing OFF)
# ============================================================
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    packing            = False,   # <-- keep each Q&A separate
    args = SFTConfig(
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_steps        = warmup_steps,
        num_train_epochs    = num_train_epochs,
        learning_rate       = learning_rate,
        fp16                = not torch.cuda.is_bf16_supported(),
        bf16                = torch.cuda.is_bf16_supported(),
        logging_steps       = logging_steps,
        save_steps          = save_steps,
        optim               = "adamw_8bit",
        weight_decay        = 0.01,
        lr_scheduler_type   = "linear",
        seed                = seed,
        output_dir          = "outputs/stage2_instruction",
        report_to           = "none",
    ),
)
print("SFTTrainer ready | packing=False | samples:", len(dataset))

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/501 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
SFTTrainer ready | packing=False | samples: 501


In [10]:
# ============================================================
# Step 8. Train  (10-15 min on a T4)
# ============================================================
print("Starting Stage 2 training...")
stats = trainer.train()

print("\nTraining complete.")
print(f"Runtime   : {stats.metrics['train_runtime']:.0f} s")
print(f"Final loss: {stats.metrics['train_loss']:.4f}")

Starting Stage 2 training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 501 | Num Epochs = 3 | Total steps = 189
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
20,3.264328
40,2.003563
60,1.731604
80,1.606655
100,1.558006
120,1.522943
140,1.413950
160,1.357764
180,1.377842


Unsloth: Restored added_tokens_decoder metadata in outputs/stage2_instruction/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/stage2_instruction/checkpoint-189/tokenizer_config.json.



Training complete.
Runtime   : 780 s
Final loss: 1.7412


## 9. Save, merge, and push to the Hub

Same pattern as Stage 1: fold the adapter into the Stage-1 weights and push one standalone model. **Stage 3 loads this repo.**

In [11]:
# ============================================================
# Step 9. Save adapter, merge, push
# ============================================================
model.save_pretrained("stage2_lora_adapter")
tokenizer.save_pretrained("stage2_lora_adapter")
print("Adapter saved -> stage2_lora_adapter/")

print("\nMerging...")
model.save_pretrained_merged("stage2_merged_model", tokenizer, save_method="merged_16bit")
print("Merged -> stage2_merged_model/")

print(f"\nPushing to {STAGE2_REPO} ...")
model.push_to_hub_merged(STAGE2_REPO, tokenizer, save_method="merged_16bit", private=PRIVATE)
print(f"Done -> https://huggingface.co/{STAGE2_REPO}")

Unsloth: Restored added_tokens_decoder metadata in stage2_lora_adapter/tokenizer_config.json.


Adapter saved -> stage2_lora_adapter/

Merging...


Unsloth: Restored added_tokens_decoder metadata in stage2_merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `stage2_merged_model`: 100%|██████████| 1/1 [00:34<00:00, 34.35s/it]


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-stage2/model.safetensors:   1%|          | 15.9MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:53<00:00, 113.14s/it]


Unsloth: Merge process complete. Saved to `/content/Mohan143/hr-policy-assistant-stage2`
Done -> https://huggingface.co/Mohan143/hr-policy-assistant-stage2


## 10. After — the same questions, now answered

Compare these against the 'before' block above. This is the payoff of Stage 2.

In [12]:
# ============================================================
# Step 10. AFTER training: same questions, same template
# ============================================================
FastLanguageModel.for_inference(model)

print("="*80)
print("AFTER STAGE 2 - instruction-tuned model")
print("="*80)
for q in test_questions:
    inputs = tokenizer(build_prompt(q), return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0], skip_special_tokens=True).split("assistant\n")[-1].strip()
    print(f"\nQ: {q}\nA: {resp[:320]}")
    print("-"*80)

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER STAGE 2 - instruction-tuned model


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How many casual leaves can I take per year?
A: The company reserves the right to apply leave without pay in situations of severe organizational need where no casual or paid leave is available. Leave without pay is deducted at full salary rate for the period. Apply for leave without pay only when all other paid leaves have been exhausted..Disclaimer: Consult HR for 
--------------------------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the work from home policy?
A: Laptop returns are handled through the IT support team. The return must be completed with a return authorization form, original receipt, and full warranty service tag. The return must be in good working condition, and refurbished devices must undergo a 9
--------------------------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How do I apply for sick leave?
A: Employees must complete
--------------------------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What benefits does the company provide?
A: Apply for maternity leave through the HR portal during your pregnancy. Medical certificates are required from the first and last month of pregnancy. You may also request leave through the leave application
--------------------------------------------------------------------------------

Q: What is the notice period for resignation?
A: Medical leave of 1-5
--------------------------------------------------------------------------------


## Understanding the loss function (read this — it's the whole game)

The number printed during training is the **cross-entropy loss** of next-token
prediction. Everything in Stages 1 and 2 is optimizing exactly this.

### What the model actually does

At every position, the model outputs a **probability for every token in the
vocabulary** — "what comes next?" The loss asks one question:

> **What probability did the model assign to the token that actually came next?**

```text
Loss for one token = -log( probability assigned to the correct token )
```

The minus-log is the key. Look at how it behaves:

| Model's probability for the correct token | Loss | Meaning |
|---|---|---|
| 1.00 (certain, correct) | 0.00 | Perfect — no penalty |
| 0.50 | 0.69 | Unsure |
| 0.10 | 2.30 | Mostly wrong |
| 0.01 | 4.61 | Confidently wrong — heavily punished |

So the loss is **small when the model is confidently right**, and **explodes when
the model is confidently wrong**. The total loss is this value averaged over
every token in the batch. Training nudges the weights to push the correct token's
probability up.

### Reading your numbers

- **Loss falling** → the model is assigning higher probability to real HR policy
  text. It's learning.
- **Loss flat from step 1** → learning rate too low, or data/format broken.
- **Loss spiking / NaN** → learning rate too high.
- **Training loss falls but validation loss rises** → **overfitting**: it's
  memorizing your examples instead of learning patterns. Use fewer epochs or
  more data.

### A useful intuition: perplexity

```python
perplexity = exp(loss)
```

Perplexity is "how many tokens is the model effectively torn between?"
Loss 2.30 → perplexity 10 → it's about as confused as if it were guessing among
10 options. **Lower is better.** A loss around 1.0–2.0 on domain text is
typically healthy for this kind of small-model fine-tune.

> ⚠️ **Low loss ≠ good model.** Loss only measures "did it predict the next token
> of *my training text*". It does not measure whether an answer is *correct*.
> That's why we test with real questions — and why Stage 3 exists.

### In Stage 2 specifically

Same cross-entropy loss — but now the tokens being predicted are the tokens of a
**human-written ideal answer**. So the loss is measuring: *how closely does the
model reproduce the answer a human would have given?*

Two Stage-2-specific things to watch:

- Stage 2 loss usually starts **lower** than Stage 1's, because the model already
  knows the domain vocabulary. That's Stage 1 paying off.
- With only a few hundred examples and 3 epochs, **overfitting is a real risk** —
  the model can memorize your exact answers. If it parrots training answers
  word-for-word to slightly different questions, reduce epochs or add data.

> Remember: this loss rewards *matching your gold answer*. It has no concept of
> "helpful", "professional", or "safe". Fixing *that* is exactly what Stage 3
> (DPO) is for.

---

## Stage 2 complete

**Next:** open **`Stage3_DPO_Alignment.ipynb`**.